In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from mlcore.tabular.preprocessing import normalize_missing_values, drop_rows_where_all_columns_missing, exclude_columns
from mlcore.tabular.analysis import gain_ratio_by_feature, absolute_correlation_matrix, descriptive_statistics, quartile_summary
from mlcore.tabular.plotting import plot_correlation_heatmap, plot_feature_scores

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 180)

In [3]:
ROOT = Path.cwd()
if not (ROOT / "mlcore").exists():
    for p in [ROOT.parent, ROOT.parent.parent, ROOT.parent.parent.parent]:
        if (p / "mlcore").exists():
            ROOT = p
            break

In [4]:
LAB_DIR = ROOT / "labs/lr-1"
DATA_PATH = LAB_DIR / "data/dataset.csv"
ASSETS_DIR = LAB_DIR / "assets"
ASSETS_DIR.mkdir(parents=True, exist_ok=True)
ROOT
DATA_PATH

PosixPath('/home/zevtos/projects/itmo/sem6/ml/labs/lr-1/data/dataset.csv')

In [5]:
df = pd.read_csv(
    DATA_PATH,
    sep=",",
    decimal=",",
    header=1,
    skiprows=[2],
    na_values=["-",""]
)

df.shape

(185, 34)

In [6]:
df.head()

,Unnamed: 0,Unnamed: 1,Глубина манометра,Dшт,Руст,Рзаб,Pлин,Руст.1,Рзаб.1,Рлин,Туст,Тна шлейфе,Тзаб,Tлин,Дебит газа,Дебит ст. конд.,Дебит воды,Дебит смеси,Дебит гааз,Дебит кон нестабильный,Дебит воды.1,Нэф,Рпл. Тек (послед точка на КВД),Рпл. Тек (Расчет по КВД),Рпл. Тек (Карноухов),Pсб,Pсб.1,Ro_g,Ro_c,Ro_w,Удельная плотность газа,G_total,КГФ,КГФ.1
0,804,05/06/08,"3576,30",7.94,249.6,370.1,101.8,249.0,359.6,101.8,53.0,"31,70",103.2,32.5,214.70,83.6,0.4,231.25,1610.37,131.3,0.4,56.8,45.25,56.5,NaN,93.6,92.38,0.806,801.0,1000.0,0.6694,2.78,311.91,NaN
1,804,06/06/08,"3576,30",9.53,233.5,364.6,101.3,231.0,338.1,102.4,58.8,"37,60",103.0,38.6,290.59,104.7,1.4,309.00,2310.23,158.5,1.4,56.8,45.25,56.5,NaN,92.9,91.69,0.806,801.0,1000.0,0.6694,3.70,288.60,NaN
2,804,07/06/08,"3576,30",11.11,213.4,357.1,101.6,211.0,314.8,100.6,63.6,"42,80",102.6,43.4,368.04,114.3,1.9,388.11,3039.49,172.3,1.9,56.8,45.25,56.5,NaN,91.4,90.20,0.806,801.0,1000.0,0.6694,4.52,248.79,NaN
3,804,08/06/08,"3576,30",12.70,191.6,347.4,98.4,187.0,291.5,99.0,64.7,"46,20",102.0,46.1,434.66,121.3,3.3,455.21,3824.08,181.5,3.3,56.8,45.25,56.5,NaN,89.2,88.03,0.806,801.0,1000.0,0.6694,5.22,223.56,NaN
4,804,09/06/08,"3576,30",14.29,171.9,337.7,99.2,169.0,270.3,99.5,64.0,"49,40",104.4,49.9,483.28,129.8,4.6,504.59,4299.10,190.6,4.7,56.8,45.25,56.5,NaN,89.7,88.53,0.806,801.0,1000.0,0.6694,5.77,215.15,NaN


In [7]:
df.columns.tolist()

['Unnamed: 0',
 'Unnamed: 1',
 'Глубина манометра',
 'Dшт',
 'Руст',
 'Рзаб',
 'Pлин',
 'Руст.1',
 'Рзаб.1',
 'Рлин',
 'Туст',
 'Тна шлейфе',
 'Тзаб',
 'Tлин',
 'Дебит газа',
 'Дебит ст. конд.',
 'Дебит воды',
 'Дебит смеси',
 'Дебит гааз',
 'Дебит кон нестабильный',
 'Дебит воды.1',
 'Нэф',
 'Рпл. Тек (послед точка на КВД)',
 'Рпл. Тек (Расчет по КВД)',
 'Рпл. Тек (Карноухов)',
 'Pсб',
 'Pсб.1',
 'Ro_g',
 'Ro_c',
 'Ro_w',
 'Удельная плотность газа ',
 'G_total',
 'КГФ',
 'КГФ.1']

In [8]:
TARGETS = ["G_total", "КГФ"]
df[TARGETS].describe(include="all")

,G_total,КГФ
count,23.000000,23.000000
mean,5.743043,265.027826
std,2.113176,40.997470
min,2.780000,199.630000
25%,4.385000,237.545000
50%,5.250000,255.680000
75%,6.125000,288.890000
max,10.930000,385.420000


In [9]:
df.dtypes.to_frame("dtype").head(20)

,dtype
Unnamed: 0,int64
Unnamed: 1,str
Глубина манометра,str
Dшт,float64
Руст,float64
Рзаб,float64
Pлин,float64
Руст.1,float64
Рзаб.1,float64
Рлин,float64


In [ ]:
df_norm = normalize_missing_values(df)
missing_table = df_norm.isna().sum().sort_values(ascending=False).to_frame("na_count")
missing_table["na_pct"] = 